# 1.5 家庭厨房助手（练习）

本练习实现一个智能家庭厨房助手，具备以下功能：

1. **识别食材**：接受文字或图片形式的食材输入
2. **搜索食谱**：使用 Tavily 搜索适合当前食材的食谱
3. **记忆偏好**：记录用户的历史饮食偏好，综合推荐个性化食谱

## 1. 环境初始化

In [1]:
from dotenv import load_dotenv

load_dotenv(override=True)

True

## 2. 工具定义

### 2.1 食谱搜索工具（Tavily）

In [2]:
from langchain.tools import tool
from tavily import TavilyClient

tavily_client = TavilyClient()

@tool
def search_recipes(query: str) -> str:
    """搜索适合特定食材的食谱。输入中文或英文搜索关键词（如食材名称、菜系），返回相关食谱信息。"""
    results = tavily_client.search(query, max_results=5)
    # 整理搜索结果，提取标题和内容摘要
    output = []
    for r in results.get("results", []):
        output.append(f"**{r['title']}**\n{r['content']}\n来源: {r['url']}")
    return "\n\n".join(output) if output else "未找到相关食谱"

### 2.2 偏好记忆工具

用简单的内存列表存储用户偏好，Agent 可以主动调用工具保存和查询偏好。

In [3]:
# 用户偏好存储（本 session 内持久化）
user_preferences: list[str] = []

@tool
def save_preference(preference: str) -> str:
    """保存用户的饮食偏好。当用户提到口味喜好（如喜欢辣、偏爱清淡）、
    忌口（如不吃香菜、对海鲜过敏）、喜爱菜系等信息时调用此工具记录下来。"""
    user_preferences.append(preference)
    return f"已记录偏好：{preference}（当前共 {len(user_preferences)} 条偏好）"

@tool
def get_preferences() -> str:
    """查询用户所有已记录的饮食偏好，在推荐食谱前调用以了解用户喜好。"""
    if not user_preferences:
        return "尚未记录任何偏好"
    prefs = "\n".join(f"  - {p}" for p in user_preferences)
    return f"用户已记录的饮食偏好（共 {len(user_preferences)} 条）：\n{prefs}"

## 3. 系统提示词

In [5]:
system_prompt = """
你是一位专业的家庭厨房助手，擅长根据现有食材推荐美味食谱。

## 你的工作流程

1. **了解食材**：用户会通过文字描述或图片展示厨房中的现有食材，仔细识别所有可用食材。

2. **查询偏好**：在推荐食谱之前，先调用 `get_preferences` 工具查询用户的历史偏好，
   确保推荐符合用户口味。

3. **搜索食谱**：使用 `search_recipes` 工具搜索适合这些食材的食谱。
   搜索时结合食材名称和用户偏好（如菜系、口味）构建精准的搜索词。

4. **个性化推荐**：综合食材、用户偏好和搜索结果，给出 2-3 个食谱推荐，并说明：
   - 食谱名称和大致做法
   - 为什么这道菜适合用户
   - 需要额外购买的食材（如果有）

5. **记录偏好**：当用户表达任何饮食偏好时（例如：喜欢辣、不吃香菜），
   立即调用 `save_preference` 工具保存，方便下次推荐时使用。

请用友好、专业的中文回答，推荐的食谱尽量家常实用。
"""

## 4. 创建 Agent

使用 `create_agent` 创建支持多轮对话的 Agent，通过 `InMemorySaver` 保存对话历史。

In [9]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

# 使用支持视觉（vision）的模型，以便处理图片食材输入
# claude-3-5-haiku 性价比高，支持图片理解
agent = create_agent(
    model="gpt-5-nano",
    tools=[search_recipes, save_preference, get_preferences],
    system_prompt=system_prompt,
    checkpointer=InMemorySaver()  # 记录对话历史，支持多轮对话
)

print("厨房助手已就绪！")

厨房助手已就绪！


## 5. 辅助函数

In [7]:
import base64
from pathlib import Path
from langchain_core.messages import HumanMessage

def chat(text: str, thread_id: str = "user_1") -> str:
    """文字输入：向厨房助手发送文字消息"""
    config = {"configurable": {"thread_id": thread_id}}
    response = agent.invoke(
        {"messages": [HumanMessage(content=text)]},
        config
    )
    return response["messages"][-1].content

def chat_with_image(image_path: str, text: str = "请识别图片中的食材，并推荐适合的食谱",
                    thread_id: str = "user_1") -> str:
    """图片输入：上传本地食材图片，向厨房助手咨询食谱"""
    image_data = base64.b64encode(Path(image_path).read_bytes()).decode()
    suffix = Path(image_path).suffix.lstrip(".").replace("jpg", "jpeg")
    message = HumanMessage(content=[
        {"type": "image", "source": {
            "type": "base64",
            "media_type": f"image/{suffix}",
            "data": image_data
        }},
        {"type": "text", "text": text}
    ])
    config = {"configurable": {"thread_id": thread_id}}
    response = agent.invoke({"messages": [message]}, config)
    return response["messages"][-1].content

def chat_with_image_url(image_url: str, text: str = "请识别图片中的食材，并推荐适合的食谱",
                        thread_id: str = "user_1") -> str:
    """图片 URL 输入：通过图片链接向厨房助手咨询食谱"""
    message = HumanMessage(content=[
        {"type": "image", "source": {
            "type": "url",
            "url": image_url
        }},
        {"type": "text", "text": text}
    ])
    config = {"configurable": {"thread_id": thread_id}}
    response = agent.invoke({"messages": [message]}, config)
    return response["messages"][-1].content

print("辅助函数已定义")

辅助函数已定义


## 6. 演示：文字输入食材

In [10]:
# 演示 1：用文字描述厨房里的食材
reply = chat("我冰箱里有：鸡胸肉、西兰花、胡萝卜、大蒜、生姜、鸡蛋。请帮我推荐几道可以做的菜。")
print(reply)

太好了，你的冰箱里这几样材料非常适合做几道家常快手菜。基于你现有的食材，给你3道可选的简单做法，步骤都不复杂，口味也比较家常。

1) 鸡胸肉炒西兰花胡萝卜（姜蒜香味版）
- 大致做法
  - 鸡胸肉切丁，用少量料酒、酱油和淀粉腌15–20分钟。
  - 锅中热油，先爆香姜末和蒜末。
  - 下鸡肉丁炒至变色，加入胡萝卜片翻炒1–2分钟。
  - 加西兰花，视需要加一点水焖1分钟，让蔬菜熟但仍保持脆感。
  - 调味：酱油、蚝油、少许糖，快炒勾薄芡出锅。
- 为什么适合你
  - 充分利用你所有的肉和蔬菜，口味鲜美且热量相对低；做法简单，家常感强。
- 需要额外购买的食材
  - 蚝油、淀粉（玉米淀粉即可）、酱油、少量糖。若没有蚝油也可用酱油+一点点盐和糖替代，口味会清淡些。

2) 西兰花胡萝卜炒鸡蛋（营养快手版）
- 大致做法
  - 鸡蛋打散，锅中油热后炒成碎块，盛出备用。
  - 同一锅中爆香姜蒜，加入胡萝卜和西兰花快速翻炒。
  - 加入适量盐和少许鸡汤或清水，蔬菜略软后回锅炒蛋碎混合均匀，快速出锅。
- 为什么适合你
  - 只用你现有的材料即可完成，蛋白质丰富，颜色亮丽，做法极简。
- 需要额外购买的食材
  - 油、盐、可选酱油；如果想要更有风味，可以加一点点胡椒和芝麻油，味道会更香。

3) 蒜香姜汁煎鸡胸配西兰花胡萝卜（简易日式风）
- 大致做法
  - 鸡胸肉拍薄，煎至两面金黄，用蒜末和姜汁（现磨姜末+少量水的混合汁）淋在肉上提香。
  - 西兰花和胡萝卜焯水后快速翻炒，盐与胡椒调味，出锅前淋少许香油。
- 为什么适合你
  - 鸡肉多汁、蔬菜爽脆，口味偏清新，适合想要简单日式风味的时候。
- 需要额外购买的食材
  - 蒜末、姜汁、香油、盐、胡椒；姜和蒜你已经有，但如想更浓郁可多准备些。

如果你愿意，我可以把其中1道改成“零油版”或“低盐版”，或者把三道菜再细化成逐步口袋菜单（按你家常饭量给出用量）。此外，现在你也可以告诉我你偏好口味（例如偏辣、偏清淡、需要无香菜等），我会据此再给出更贴合你口味的1–2道候选。需要我把以上菜谱的用量和分步清单再整理成简短购物清单吗？


In [11]:
# 演示 2：用户表达个人偏好（Agent 会自动保存）
reply = chat("我平时不喜欢吃太油腻的，偏爱清淡健康的口味，另外我对香菜过敏。")
print(reply)

谢谢你更新偏好信息。我已经记录为：偏好清淡健康、低油少脂，避免香菜，偏好姜蒜香味。下面给出3道更贴合你口味且尽量低油的做法，都用到你现有的材料，且不含香菜。

1) 低油版 鸡胸肉炒西兰花胡萝卜（姜蒜香）
- 核心做法要点
  - 鸡胸肉切丁，轻轻腌15分钟（可用少量生抽、胡椒），用不粘锅或少量植物油（约1茶匙）快速翻炒至变色。
  - 姜末、蒜末先爆香，加入肉丁，再放胡萝卜片、西兰花，快速翻炒1–2分钟。
  - 加一点点水焖1分钟，保持蔬菜脆感；用少量生抽、蚝油（可选）和糖调味，收汁即可。
- 为什么符合偏好
  - 低油、快速高温短炒，保留蔬菜口感与营养，香气来自姜蒜，口味清淡但有层次。
- 需要额外购买的食材（如需）
  - 生抽、蚝油（可选）、淀粉（用来薄薄勾芡，可省略）。若尽量简化，可只用盐和一点点糖调味。

2) 蛋香西兰花胡萝卜炒蛋（蛋白质+蔬菜，低油快手）
- 核心做法要点
  - 鸡蛋打散，锅中少油煎成大蛋块，盛出备用。
  - 同锅爆香姜蒜，加入胡萝卜片和西兰花，快速翻炒；加少量水焖软。
  - 回锅把蛋块倒回，快速翻匀，蛋香混合蔬菜香气，出锅前可滴几滴香油（可选）。
- 为什么符合偏好
  - 蛋白质丰富，油量很低，蔬菜保持脆甜，口味清淡健康，香气来自姜蒜。
- 需要额外购买的食材（如需）
  - 油、盐；酱油可选，香油少许（可省略）。

3) 蒜香姜汁煎鸡胸，清炒蔬菜（无香菜版）
- 核心做法要点
  - 鸡胸肉拍薄，平底锅少油煎至两面金黄，切片备用。
  - 锅内留少量油，加入姜丝和蒜末炒出香味，淋入少量热水形成轻薄姜汁。
  - 西兰花和胡萝卜用清水焯一下或快速翻炒，盐、胡椒调味，出锅前滴几滴香油（可选）。
  - 将鸡胸肉片淋上姜汁，搭配蔬菜一起吃。
- 为什么符合偏好
  - 少油、清淡，姜蒜香味突出，避免香菜，做法简单且风味清爽。
- 需要额外购买的食材（如需）
  - 香油（可选）、盐、胡椒；姜、蒜本身就有，且你有。

如果你愿意，我可以把这三道菜各自的用量按两人份给出详细步骤和逐步分量（例如鸡胸肉300–350克、西兰花1朵等），也可以把它们整理成一张简短的购物清单或逐日菜谱。请告诉我你更偏向改成哪一种版本。


In [12]:
# 验证：查看 Agent 是否记住了偏好
print("当前已记录的用户偏好：")
print(get_preferences.invoke({}))

当前已记录的用户偏好：
用户已记录的饮食偏好（共 1 条）：
  - 偏好清淡健康、低油少脂；香菜过敏，避免香菜；口味偏清淡、偏好姜蒜香味


## 7. 演示：图片输入食材

通过图片 URL 展示厨房食材，Agent 会识别食材并推荐食谱。

In [16]:
# 演示 3：通过图片 URL 识别食材
# 这里使用一张示例蔬菜图片，实际使用时替换为你的食材照片 URL
sample_image_url = "https://media-cldnry.s-nbcnews.com/image/upload/t_fit-560w,f_auto,q_auto:best/newscms/2016_17/1068426/fridge-inline-today-160428.JPG"

reply = chat_with_image_url(
    image_url=sample_image_url,
    text="这是我厨房里的食材，请识别图片中有哪些食材，并结合我的饮食偏好推荐合适的食谱",
    thread_id="user_1"  # 使用同一个 thread_id，Agent 会记住之前的偏好
)
print(reply)

很棒的组合！我先把你图片中大致能辨识的食材和你这次新买的材料整理一下，然后给出2–3道符合你偏好的清淡低油、姜蒜香味为主的做法。

一、从图片中能看到的大致食材（供你快速确认）
- 彩椒：红、黄、橙各一个，颜色很亮，炒菜提香很棒。
- 生菜叶/莴苣类蔬菜一把，叶片新鲜。
- 黄瓜若干（看起来是小黄瓜，口感清脆）。
- 鸡蛋一盒/ carton（数量看起来不少）。
- 奶酪块和牛奶，属于奶制品备料。
- 碗里放着的红色果实，可能是樱桃番茄或草莓（看起来很甜美的水果球）。
- 可能还有一些根茎类蔬菜（图片左下方的白色细长茎状，可能是白芦笋或芹菜）。
- 你新买的材料：豆腐、菠菜、蘑菇。

二、结合偏好（清淡、低油、无香菜、偏姜蒜香味）的推荐
1) 鸡胸肉+豆腐+菠菜+蘑菇姜蒜炒
- 用材：鸡胸肉、豆腐、菠菜、蘑菇、大蒜、姜、少量油、酱油、盐、可选蚝油
- 做法要点：
  - 鸡胸肉切薄片，姜蒜切末，少量油热锅爆香姜蒜。
  - 先下鸡肉片快炒至变色，盛出备用。
  - 锅中再放蘑菇快炒出香味，加入豆腐块与菠菜，翻炒均匀。
  - 回锅加入鸡肉，调味（生抽、盐、少量糖、蚝油可选），快速翻炒一下即可出锅。
- 为什么符合偏好：全程低油快速翻炒，姜蒜香味突出，蔬菜和蛋白质搭配均衡，口味清淡但有层次。
- 额外购买需求：生抽、蚝油（可选，若不想用也可用盐+糖替代）。

2) 菠菜蘑菇豆腐汤（蛋花或清汤版）
- 用材：菠菜、蘑菇、豆腐、姜、蒜、鸡蛋（可选）、盐、清汤或水
- 做法要点：
  - 锅中加水或清汤，放入姜蒜煮出香味。
  - 依次放入蘑菇、豆腐，煮几分钟至入味。
  - 加入菠菜，煮至菠菜软熟。
  - 若要蛋花：打散鸡蛋，缓慢倒入锅中成蛋花；不吃蛋也可直接关火，调味即可。
- 为什么符合偏好：清淡、低油，蛋花或蛋香增蛋白质，姜蒜香气突出。
- 额外购买需求：盐、香油（可选）。

3) 彩椒豆腐煎配蒜香菠菜
- 用材：彩椒、豆腐、菠菜、大蒜、姜、少量油
- 做法要点：
  - 豆腐切厚块，小火煎至两面微黄，盛出备用。
  - 彩椒切块，蒜和姜切末，锅中少油快速翻炒彩椒至断生，加入菠菜翻几下。
  - 将煎好的豆腐回锅，快速翻匀，调味（盐、少量酱油即可）。
- 为什么符合偏好：颜色丰富且口味清淡，姜蒜香味来自蒜姜，油用量可控制在最低限度。
- 额外购买需求：盐、酱油（如

In [ ]:
# 演示 4：本地图片输入（将文件路径替换为你的实际图片路径）
# 取消注释并修改路径即可使用

# local_image_path = "/path/to/your/kitchen_ingredients.jpg"
# reply = chat_with_image(
#     image_path=local_image_path,
#     text="图片是我厨房里现有的食材，请推荐今晚可以做什么菜？",
#     thread_id="user_1"
# )
# print(reply)

print("（取消上方注释并设置本地图片路径即可测试本地图片输入）")

## 8. 演示：历史偏好的跨对话记忆

新用户（新 thread_id）第一次对话时，Agent 会发现没有历史偏好；
同一用户（相同 thread_id）再次对话时，Agent 会自动应用之前记录的偏好。

In [13]:
# 演示 5：新用户，没有历史偏好
reply_new_user = chat(
    text="我有番茄、鸡蛋、葱，能做什么？",
    thread_id="new_user_001"  # 新的 thread_id，没有历史记录
)
print("=== 新用户（无历史偏好）===")
print(reply_new_user)

=== 新用户（无历史偏好）===
太好了，你手头只有番茄、鸡蛋和葱，也能做出几道简单又家常的菜。结合你偏好清淡、低油、喜姜蒜香的口味，给你 3 个推荐：

1) 番茄炒蛋（清淡版，带姜蒜香）
- 大致做法：番茄切块，葱切葱花，蒜和姜切末（可选）。锅里用少量油先煸香蒜姜和葱白，然后加入番茄炒出汁，再把打散的鸡蛋倒入翻炒成蛋块，最后用盐略调味，撒上葱绿出锅。
- 为什么适合你：经典家常，快速且口味清淡，姜蒜香可以提升层次，但油用量很少，容易做到低脂。
- 需要额外购买的食材（如果你家里已有就不需要）：油、盐、糖（帮助平衡酸味），姜和蒜（可选，若没有也可以不放）。

2) 西红柿蛋花汤
- 大致做法：葱姜蒜用少量油煸香，加入切块的番茄煸出香味和汁水，然后加水煮开，缓慢倒入打散的鸡蛋，边倒边用筷子搅拌成蛋花，最后加盐、糖调味，撒葱花即可。
- 为什么适合你：汤品清淡、营养丰富，油脂更低，香气来自姜蒜和葱花，做起来也很快。
- 需要额外购买的食材：水或高汤、盐、糖；油和姜蒜可选，若你不想用油也可以用不粘锅“干煸”香味。

3) 番茄蒸蛋（番茄蛋羹）
- 大致做法：番茄切丁，鸡蛋打散后与番茄丁、葱花、少量水混合均匀，加入盐和姜蒜末（可选），装进耐热碗中蒸约8–10分钟，蒸好后可淋一点点香油提香。
- 为什么适合你：口感柔软、蛋香和番茄香混合，蒸制法对油的依赖更低，整体更轻盈。
- 需要额外购买的食材：蒸锅或能用来蒸的器皿、水；盐、姜蒜末如果没有也可以不放。

如果你愿意，我还能把其中一两道再按你现有的调味品做法细化成逐步图解，或者根据你手边的调料（比如有没有香油、糖、盐、鸡精等）再给出更精准的配方。还想要我把你最近的偏好再保存一下吗？比如“更偏好姜蒜香、低油”之类的，以便下次推荐更贴合。


In [14]:
# 演示 6：回到老用户，Agent 记住了清淡口味和香菜过敏的偏好
reply_returning = chat(
    text="今天又买了些食材：豆腐、菠菜、蘑菇。结合上次的食材，有什么好推荐的吗？",
    thread_id="user_1"  # 回到之前的对话线程
)
print("=== 老用户（有历史偏好）===")
print(reply_returning)

=== 老用户（有历史偏好）===
太好了，新添的豆腐、菠菜、蘑菇和你手头的材料可以组合出更多清淡健康的菜式。下面给你3道尽量低油、姜蒜香味为主、且不含香菜的选项，便于快速上手。

1) 低油版 豆腐蘑菇菠菜鸡胸肉炒
- 主要材料：鸡胸肉、豆腐、蘑菇、菠菜、大蒜、姜
- 做法要点：
  - 鸡胸肉切薄片，姜蒜切末，用少量油快速炒至变色，盛出备用。
  - 同锅加入蘑菇快炒出香味，再放豆腐块和菠菜，翻炒至菠菜略软。
  - 加回鸡肉，调入生抽、少许蚝油（可选）、少量糖做提鲜，快速翻匀后出锅。
- 为什么适合你：全程低油，蛋白质来自鸡肉和豆腐，蔬菜新鲜，口味清淡但层次感强，姜蒜香味突出。
- 额外购买需求：生抽、蚝油（可选，若不想用也可用盐+一点糖替代）。

2) 蘑菇菠菜豆腐蛋花汤
- 主要材料：豆腐、菠菜、蘑菇、姜、蒜、鸡蛋
- 做法要点：
  - 锅中加水或清汤，放入姜蒜煮出香味。
  - 加入蘑菇煮软，放入豆腐小块，最后缓缓倒入打散的蛋液成蛋花。
  - 加入菠菜，调盐或少许酱油，出锅前可加几滴香油提香（可选）。
- 为什么适合你：清淡、低油、汤品易消化，蛋花增加蛋白质；香菜天然避免，口味以姜蒜香为主。
- 额外购买需求：蛋、盐、香油（可选）。

3) 菠菜蘑菇豆腐西兰花（加蛋的变体）
- 主要材料：菠菜、蘑菇、豆腐、西兰花、鸡蛋、大蒜、姜
- 做法要点：
  - 蘑菇和豆腐切块，西兰花焯水（可选）快速翻炒，加入菠菜至叶子变软。
  - 另起一锅，打散的蛋液快速炒成蛋块，随后加入锅中蔬菜，拌匀。
  - 用姜蒜爆香后，加入所有材料，调味用盐和少许酱油，最后快速翻炒2分钟即可。
- 为什么适合你：把豆腐、菠菜、蘑菇和蛋结合，口味清淡，蛋香和姜蒜香并存，整体油脂很低。
- 额外购买需求：盐、酱油、香油（可选）。

如果你愿意，我可以把这三道菜都细化成两人份的用量和逐步清单，或把其中1–2道整理成“零油版/低盐版”的版本。也可以根据你今天的实际就餐人数，给出具体的分量表和购物清单。需要我把用量写成两个版本（2人份和4人份）吗？另外，告诉我你今晚更想吃哪一道，我可以先给出详细的步骤和用量。


In [15]:
# 查看整个对话历史（了解 InMemorySaver 的工作方式）
from pprint import pprint

config = {"configurable": {"thread_id": "user_1"}}
state = agent.get_state(config)
print(f"对话历史共 {len(state.values['messages'])} 条消息")
print(f"当前记录的用户偏好：{user_preferences}")

对话历史共 21 条消息
当前记录的用户偏好：['偏好清淡健康、低油少脂；香菜过敏，避免香菜；口味偏清淡、偏好姜蒜香味']
